In [1]:
import re, gc, os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = torch.device("cuda")
MAX_LEN, BATCH, ACCUM, EPOCHS, LR = 512, 8, 2, 3, 2e-5
tok = AutoTokenizer.from_pretrained("microsoft/codebert-base")
RESULTS = "../results/codebert_cross_project.csv"
os.makedirs("../results", exist_ok=True)

def clean_code(code):
    code = re.sub(r"/\*.*?\*/", " ", code, flags=re.DOTALL)
    code = re.sub(r"//[^\n]*", " ", code)
    code = re.sub(r"\s+", " ", code)
    return code.strip()

df = pd.read_json("../data/function.json")
df["clean"] = df["func"].apply(clean_code)
qemu = df[df["project"] == "qemu"]
ffmpeg = df[df["project"] == "FFmpeg"]

def make_loader(d, shuffle):
    items = list(zip(d["clean"], d["target"]))
    def collate(batch):
        texts, labels = zip(*batch)
        enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor(labels)
        return enc
    return DataLoader(items, batch_size=BATCH, shuffle=shuffle, collate_fn=collate)

def run(train_df, test_df, name):
    torch.manual_seed(42); np.random.seed(42)
    train_dl, test_dl = make_loader(train_df, True), make_loader(test_df, False)
    model = AutoModelForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2).to(device)
    counts = np.bincount(train_df["target"].values)
    weights = torch.tensor(len(train_df) / (2 * counts), dtype=torch.float32, device=device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * len(train_dl) // ACCUM
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for i, b in enumerate(train_dl):
            b = {k: v.to(device) for k, v in b.items()}
            with torch.amp.autocast("cuda"):
                logits = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits
                loss = loss_fn(logits.float(), b["labels"]) / ACCUM
            scaler.scale(loss).backward()
            if (i + 1) % ACCUM == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
            if i % 500 == 0:
                print(f"[{name}] epoch {epoch} step {i}/{len(train_dl)} loss {loss.item() * ACCUM:.3f}")

    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for b in test_dl:
            b = {k: v.to(device) for k, v in b.items()}
            with torch.amp.autocast("cuda"):
                logits = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits
            preds.append(logits.argmax(1).cpu().numpy())
            labels.append(b["labels"].cpu().numpy())
    y, p = np.concatenate(labels), np.concatenate(preds)
    res = dict(name=name, n_train=len(train_df), n_test=len(test_df),
               acc=accuracy_score(y, p), precision=precision_score(y, p, zero_division=0),
               recall=recall_score(y, p, zero_division=0), f1=f1_score(y, p, zero_division=0))
    pd.DataFrame([res]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print(res)

    del model, opt, sched, scaler, loss_fn
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
f_tr, f_te = train_test_split(ffmpeg, test_size=0.2, random_state=42, stratify=ffmpeg["target"])
q_tr, q_te = train_test_split(qemu, test_size=0.2, random_state=42, stratify=qemu["target"])

scenarios = [
    ("QEMU -> FFmpeg", qemu, ffmpeg),
    ("FFmpeg -> QEMU", ffmpeg, qemu),
    ("QEMU -> QEMU", q_tr, q_te),
    ("FFmpeg -> FFmpeg", f_tr, f_te),
]

done = set(pd.read_csv(RESULTS)["name"]) if os.path.exists(RESULTS) else set()
for name, tr, te in scenarios:
    if name not in done:
        run(tr, te, name)

print(pd.read_csv(RESULTS))

In [2]:
gt = pd.read_json("../data/grouped_train.jsonl", lines=True, dtype={"commit_id": str})
ge = pd.read_json("../data/grouped_test.jsonl", lines=True, dtype={"commit_id": str})
for d in (gt, ge):
    d["clean"] = d["func"].apply(clean_code)
run(gt, ge, "CodeBERT grouped split")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[CodeBERT grouped split] epoch 1 step 0/2737 loss 0.807
[CodeBERT grouped split] epoch 1 step 500/2737 loss 0.723
[CodeBERT grouped split] epoch 1 step 1000/2737 loss 0.522
[CodeBERT grouped split] epoch 1 step 1500/2737 loss 0.614
[CodeBERT grouped split] epoch 1 step 2000/2737 loss 0.674
[CodeBERT grouped split] epoch 1 step 2500/2737 loss 0.653
[CodeBERT grouped split] epoch 2 step 0/2737 loss 0.467
[CodeBERT grouped split] epoch 2 step 500/2737 loss 0.412
[CodeBERT grouped split] epoch 2 step 1000/2737 loss 0.608
[CodeBERT grouped split] epoch 2 step 1500/2737 loss 0.638
[CodeBERT grouped split] epoch 2 step 2000/2737 loss 0.604
[CodeBERT grouped split] epoch 2 step 2500/2737 loss 0.663
[CodeBERT grouped split] epoch 3 step 0/2737 loss 0.524
[CodeBERT grouped split] epoch 3 step 500/2737 loss 0.802
[CodeBERT grouped split] epoch 3 step 1000/2737 loss 0.657
[CodeBERT grouped split] epoch 3 step 1500/2737 loss 0.300
[CodeBERT grouped split] epoch 3 step 2000/2737 loss 0.532
[CodeBERT